In [1]:
import psycopg2
from psycopg2 import sql, extras
import logging
import os
import datetime

from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

C:\Users\johna\miniforge3\envs\thecall\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
source_dir = r"E:\Callproject\assembled"

In [3]:
DB_PARAMS = {
    'dbname': 'thecall',
    'user': 'postgres',
    'password': 'password',
    'host': 'localhost',
    'port': '5432'
}

In [4]:
def connect_to_db(params):
    """Establishes a connection to the PostgreSQL database."""
    conn = None
    try:
        logging.info("Attempting to connect to PostgreSQL database...")
        conn = psycopg2.connect(**params)
        conn.autocommit = False
        logging.info("Connection successful.")
        return conn
    except psycopg2.Error as e:
        logging.error(f"Error connecting to the database: {e}")
        return None

In [5]:
def create_table(conn):
    """
    Creates the 'filelist' table if it does not already exist.
    Drops and recreates if you want a fresh scan — comment out DROP if resuming.
    """
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS filelist (
                id        SERIAL PRIMARY KEY,
                filename  TEXT        NOT NULL,
                folder    TEXT        NOT NULL,
                filedate  TIMESTAMP,
                filetype  TEXT,
                filesize  BIGINT
            );
        """)
        conn.commit()
        logging.info("Table 'filelist' is ready.")

In [6]:
def iter_files(root_dir):
    """
    Walks the directory tree and yields one dict per file.
    Uses os.scandir for speed — much faster than os.walk on large trees.
    """
    stack = [root_dir]
    while stack:
        current = stack.pop()
        try:
            with os.scandir(current) as it:
                for entry in it:
                    if entry.is_dir(follow_symlinks=False):
                        stack.append(entry.path)
                    elif entry.is_file(follow_symlinks=False):
                        try:
                            stat = entry.stat()
                            name = entry.name
                            folder = os.path.dirname(entry.path)
                            # Use last-modified time; switch to stat.st_ctime for creation time
                            filedate = datetime.datetime.fromtimestamp(stat.st_mtime)
                            _, ext = os.path.splitext(name)
                            filetype = ext.lstrip('.').lower() if ext else None
                            filesize = stat.st_size
                            yield {
                                'filename': name,
                                'folder':   folder,
                                'filedate': filedate,
                                'filetype': filetype,
                                'filesize': filesize,
                            }
                        except OSError as e:
                            logging.warning(f"Could not stat file {entry.path}: {e}")
        except PermissionError as e:
            logging.warning(f"Permission denied: {current} — {e}")

In [7]:
def insert_batch(cur, batch):
    """Bulk-inserts a list of file-record dicts using execute_values."""
    values = [
        (r['filename'], r['folder'], r['filedate'], r['filetype'], r['filesize'])
        for r in batch
    ]
    extras.execute_values(
        cur,
        """
        INSERT INTO filelist (filename, folder, filedate, filetype, filesize)
        VALUES %s
        """,
        values,
        page_size=1000
    )

In [8]:
def main():
    BATCH_SIZE = 5000   # rows committed per transaction — tune as needed

    conn = connect_to_db(DB_PARAMS)
    if conn is None:
        logging.error("Could not establish database connection. Aborting.")
        return

    try:
        create_table(conn)

        batch = []
        total_inserted = 0

        with conn.cursor() as cur:
            # tqdm gives a live progress counter; total is approximate for display only
            for file_record in tqdm(iter_files(source_dir), desc="Scanning files", unit="file"):
                batch.append(file_record)

                if len(batch) >= BATCH_SIZE:
                    insert_batch(cur, batch)
                    conn.commit()
                    total_inserted += len(batch)
                    batch.clear()

            # Insert any remaining records
            if batch:
                insert_batch(cur, batch)
                conn.commit()
                total_inserted += len(batch)

        logging.info(f"Done. {total_inserted:,} files written to 'filelist'.")
        print(f"\nFinished. {total_inserted:,} records inserted.")

    except Exception as e:
        conn.rollback()
        logging.error(f"Unexpected error — transaction rolled back: {e}")
        raise
    finally:
        conn.close()
        logging.info("Database connection closed.")

In [9]:
if __name__ == "__main__":
    main()

2026-03-31 21:02:20,292 - INFO - Attempting to connect to PostgreSQL database...
2026-03-31 21:02:20,462 - INFO - Connection successful.
2026-03-31 21:02:20,500 - INFO - Table 'filelist' is ready.
Scanning files: 416391file [00:06, 59915.71file/s]
2026-03-31 21:02:27,467 - INFO - Done. 416,391 files written to 'filelist'.
2026-03-31 21:02:27,467 - INFO - Database connection closed.



Finished. 416,391 records inserted.
